# 01_build_startups_with_patents

**Objective:** Build the firm-level analytical table by aggregating pre-funding patent features (novelty, competitive density, OECD indicators) to the startup grain, keeping only patents with a priority date before the first equity round.

**Inputs:** `startups_master.csv`; `startup_patent_matches.csv`; `novelty_with_oecd_quality.parquet`; `lens_to_familyid.parquet`.

**Outputs:** `1_startups_with_patents.csv` (one row per startup).

## Imports

In [ ]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path

## Configuration: input/output paths and OECD metric pairs

In [ ]:
# Paths
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
MASTER_PATH   = PROC / 'startups_master.csv'
MATCHES_PATH  = PROC / 'startup_patent_matches.csv'
NOVELTY_PATH  = PROC / 'novelty_with_oecd_quality.parquet'
LENS2FAM_PATH = PROC / 'lens_to_familyid.parquet'
OUT_PATH      = PROC / '1_startups_with_patents.csv'

In [ ]:
# OECD metric pairs (EP preferred, US fallback) mapped to output columns
OECD_PAIRS = {
    'bwd_cits':     ('ep_bwd_cits',     'us_bwd_cits'),
    'npl_cits':     ('ep_npl_cits',     'us_npl_cits'),
    'originality':  ('ep_originality',  'us_originality'),
    'radicalness':  ('ep_radicalness',  'us_radicalness'),
    'patent_scope': ('ep_patent_scope', 'us_patent_scope'),
    'claims':       ('ep_claims',       'us_claims'),
    'family_size':  ('ep_family_size',  'us_family_size'),
}

## Main pipeline: load inputs, build per-patent panel, apply cutoff, aggregate, write

In [ ]:
# Main pipeline: build the per-patent panel, apply the pre-funding cutoff, aggregate to firm level, and write
def main():
    master = pd.read_csv(MASTER_PATH)
    master['first_equity_round_date'] = pd.to_datetime(
        master['first_equity_round_date'], errors='coerce'
    )

    matches = pd.read_csv(
        MATCHES_PATH,
        usecols=['Organization Name', 'lens_id', 'earliest_priority'],
    )
    matches = matches.drop_duplicates(subset=['Organization Name', 'lens_id'])
    matches['earliest_priority'] = pd.to_datetime(matches['earliest_priority'], errors='coerce')

    nv = pd.read_parquet(NOVELTY_PATH)
    l2f = pd.read_parquet(LENS2FAM_PATH)[['lens_id', 'docdb_family_id']].drop_duplicates('lens_id')

    patent_panel = matches.merge(nv, on='lens_id', how='left')
    patent_panel = patent_panel.merge(l2f, on='lens_id', how='left', suffixes=('', '_l2f'))
    if 'docdb_family_id_l2f' in patent_panel.columns:
        patent_panel['docdb_family_id'] = patent_panel['docdb_family_id'].fillna(
            patent_panel['docdb_family_id_l2f']
        )
        patent_panel = patent_panel.drop(columns=['docdb_family_id_l2f'])

    for out_col, (ep_col, us_col) in OECD_PAIRS.items():
        patent_panel[out_col] = patent_panel[ep_col].combine_first(patent_panel[us_col])

    patent_panel = patent_panel.merge(
        master[['Organization Name', 'first_equity_round_date']],
        on='Organization Name', how='left',
    )
    n_before = len(patent_panel)
    before_cutoff = patent_panel['earliest_priority'] < patent_panel['first_equity_round_date']
    patent_panel = patent_panel[before_cutoff].copy()
    print(f'Cutoff filter: kept {len(patent_panel):,} of {n_before:,} (Org, lens_id) rows '
          f'with priority_date < first_equity_round_date')

    def join_unique(s: pd.Series) -> str:
        vals = sorted({str(int(x)) if isinstance(x, (int, np.integer)) else str(x)
                       for x in s.dropna().unique()})
        return ';'.join(vals) if vals else ''

    agg = patent_panel.groupby('Organization Name').agg(
        n_patents              =('lens_id',         'nunique'),
        n_patents_with_novelty =('novelty_mean',    'count'),
        novelty_mean           =('novelty_mean',    'mean'),
        novelty_max            =('novelty_mean',    'max'),
        novelty_min_mean       =('novelty_min',     'mean'),
        novelty_top10_mean     =('novelty_top10',   'mean'),
        novelty_top10_max      =('novelty_top10',   'max'),
        bwd_cits_mean          =('bwd_cits',        'mean'),
        bwd_cits_max           =('bwd_cits',        'max'),
        npl_cits_mean          =('npl_cits',        'mean'),
        npl_cits_max           =('npl_cits',        'max'),
        originality_mean       =('originality',     'mean'),
        originality_max        =('originality',     'max'),
        radicalness_mean       =('radicalness',     'mean'),
        radicalness_max        =('radicalness',     'max'),
        claims_mean            =('claims',          'mean'),
        claims_max             =('claims',          'max'),
        family_size_mean       =('family_size',     'mean'),
        family_size_max        =('family_size',     'max'),
        patent_scope_mean      =('patent_scope',    'mean'),
        patent_scope_max       =('patent_scope',    'max'),
        lens_ids               =('lens_id',         join_unique),
        docdb_family_ids       =('docdb_family_id', join_unique),
    ).reset_index()

    out = master.merge(agg, on='Organization Name', how='left')
    out['n_patents'] = out['n_patents'].fillna(0).astype(int)
    out['n_patents_with_novelty'] = out['n_patents_with_novelty'].fillna(0).astype(int)
    out['lens_ids'] = out['lens_ids'].fillna('')
    out['docdb_family_ids'] = out['docdb_family_ids'].fillna('')

    out.to_csv(OUT_PATH, index=False)
    print(f'Wrote: {OUT_PATH}  ({out.shape[0]:,} rows x {out.shape[1]} cols)')
    print(f'  startups with >=1 matched patent : {(out["n_patents"]>0).sum():,}')
    print(f'  startups with novelty/OECD data  : {(out["n_patents_with_novelty"]>0).sum():,}')

## Run when executed as a script

In [ ]:
# Entry point
if __name__ == '__main__':
    main()